# Battery Modeling for eVTOL Analysis

This notebook demonstrates the comprehensive battery modeling capabilities in evtol-tools for eVTOL aircraft analysis. Topics covered:

- **Battery Configuration** - Series/parallel cell layout and sizing
- **Chemistry Support** - Multiple lithium-based chemistries with voltage/resistance parameters
- **Discharge Curves** - Voltage vs SoC and C-rate modeling
- **C-rate Conversions** - Bidirectional current and C-rate conversions
- **Internal Resistance** - Pack resistance from cell configuration
- **Heat Generation** - I²R power losses for thermal analysis
- **Charge Modeling** - CC-CV charging profiles with efficiency
- **Thermal Framework** - Temperature limits, derating, and thermal dynamics
- **Mission Simulation** - Combining discharge and thermal models

In [1]:
# Import battery module and supporting quantities
from evtoltools.common import (
    Mass, Capacity, Current, Voltage, Power, Energy,
    Temperature, Time, Resistance
)
from evtoltools.components import Battery
from evtoltools.components.battery import (
    # Chemistry
    LITHIUM_ION, LITHIUM_POLYMER, LITHIUM_IRON_PHOSPHATE, LITHIUM_NMC,
    get_chemistry,
    # Discharge/Charge models
    DischargeModel, AnalyticalDischargeModel, LookupTableDischargeModel,
    SimpleChargeModel, load_discharge_model,
    # Thermal
    ThermalLimits, SimpleThermalModel,
    # PyBaMM utilities
    create_default_lookup_table,
)
import numpy as np

print("Battery module imported successfully!")
print(f"Available chemistries: lithium_ion, lithium_polymer, lithium_iron_phosphate, lithium_nmc")

Battery module imported successfully!
Available chemistries: lithium_ion, lithium_polymer, lithium_iron_phosphate, lithium_nmc


---
## Part 1: Battery Configuration and Chemistry

Batteries are configured as series/parallel strings of cells. The chemistry defines voltage characteristics and internal resistance.

### 1.1 Creating a Battery Pack

In [2]:
# Create a typical eVTOL battery pack
battery = Battery(
    cells_series=14,        # 14S for ~51.8V nominal
    cells_parallel=4,       # 4P for 20Ah capacity
    cell_capacity=Capacity(5, 'Ah'),
    cell_mass=Mass(50, 'g'),
    chemistry='lithium_ion',
    c_rating_charge=1.0,    # 1C charge rate
    c_rating_discharge=2.0, # 2C discharge rate
    pack_overhead_fraction=0.15,  # 15% for BMS, wiring, enclosure
)

print(f"Battery Configuration: {battery.cells_series}S{battery.cells_parallel}P")
print(f"Total Cells: {battery.total_cells}")
print(f"Chemistry: {battery.chemistry.name}")

Battery Configuration: 14S4P
Total Cells: 56
Chemistry: lithium_ion


In [3]:
# Pack electrical properties
print("ELECTRICAL PROPERTIES")
print("=" * 50)
print(f"Nominal Voltage:    {battery.nominal_voltage}")
print(f"Max Voltage:        {battery.max_voltage}")
print(f"Min Voltage:        {battery.min_voltage}")
print(f"Voltage Range:      {battery.min_voltage.in_units_of('V'):.1f}V - {battery.max_voltage.in_units_of('V'):.1f}V")
print()
print(f"Total Capacity:     {battery.total_capacity}")
print(f"Energy Capacity:    {battery.energy_capacity}")
print(f"                    {battery.energy_capacity.in_units_of('kWh'):.3f} kWh")
print()
print(f"Pack Mass:          {battery.mass}")
print(f"                    {battery.mass.in_units_of('kg'):.2f} kg")

ELECTRICAL PROPERTIES
Nominal Voltage:    51.800000000000004 volt
Max Voltage:        58.800000000000004 volt
Min Voltage:        39.199999999999996 volt
Voltage Range:      39.2V - 58.8V

Total Capacity:     20 ampere_hour
Energy Capacity:    1036.0 watt_hour
                    1.036 kWh

Pack Mass:          3.22 kilogram
                    3.22 kg


### 1.2 Battery Sizing Methods

In [4]:
# Method 1: Size from target voltage
battery_from_v = Battery.from_target_voltage(
    target_voltage=Voltage(48, 'V'),
    cells_parallel=4,
    cell_capacity=Capacity(5, 'Ah'),
    cell_mass=Mass(50, 'g'),
    chemistry='lithium_ion'
)

print("SIZED FROM TARGET VOLTAGE (48V)")
print("=" * 50)
print(f"Configuration: {battery_from_v.cells_series}S{battery_from_v.cells_parallel}P")
print(f"Actual Voltage: {battery_from_v.nominal_voltage}")
if battery_from_v.warnings:
    print(f"Notes: {battery_from_v.warnings[0]}")

SIZED FROM TARGET VOLTAGE (48V)
Configuration: 13S4P
Actual Voltage: 48.1 volt
Notes: Rounded 12.97 cells to 13. Actual voltage: 48.1V (target was 48.0V)


In [5]:
# Method 2: Size from target energy
battery_from_e = Battery.from_target_energy(
    target_energy=Energy(5, 'kWh'),
    target_voltage=Voltage(400, 'V'),
    cell_capacity=Capacity(5, 'Ah'),
    cell_mass=Mass(70, 'g'),
    chemistry='lithium_nmc'
)

print("SIZED FROM TARGET ENERGY (5 kWh @ 400V)")
print("=" * 50)
print(f"Configuration: {battery_from_e.cells_series}S{battery_from_e.cells_parallel}P")
print(f"Total Cells: {battery_from_e.total_cells}")
print(f"Actual Energy: {battery_from_e.energy_capacity.in_units_of('kWh'):.2f} kWh")
print(f"Pack Mass: {battery_from_e.mass.in_units_of('kg'):.1f} kg")
print(f"Energy Margin: {battery_from_e.info['energy_margin_percent']:.1f}%")

SIZED FROM TARGET ENERGY (5 kWh @ 400V)
Configuration: 109S3P
Total Cells: 327
Actual Energy: 6.05 kWh
Pack Mass: 26.3 kg
Energy Margin: 21.0%


### 1.3 Battery Chemistries

In [6]:
# Compare different chemistries
chemistries = [
    ('Lithium-Ion', LITHIUM_ION),
    ('Lithium Polymer', LITHIUM_POLYMER),
    ('LiFePO4', LITHIUM_IRON_PHOSPHATE),
    ('Lithium NMC', LITHIUM_NMC),
]

print("BATTERY CHEMISTRY COMPARISON")
print("=" * 90)
print(f"{'Chemistry':<20} {'Nominal V':<12} {'Max V':<10} {'Min V':<10} {'R (mohm)':<12} {'PyBaMM Set'}")
print("-" * 90)

for name, chem in chemistries:
    r_mohm = chem.internal_resistance.in_units_of('mohm') if chem.internal_resistance else 'N/A'
    print(f"{name:<20} {chem.nominal_cell_voltage.in_units_of('V'):<12.2f} "
          f"{chem.max_cell_voltage.in_units_of('V'):<10.2f} "
          f"{chem.min_cell_voltage.in_units_of('V'):<10.2f} "
          f"{r_mohm:<12} {chem.pybamm_parameter_set}")

BATTERY CHEMISTRY COMPARISON
Chemistry            Nominal V    Max V      Min V      R (mohm)     PyBaMM Set
------------------------------------------------------------------------------------------
Lithium-Ion          3.70         4.20       2.80       30           Chen2020
Lithium Polymer      3.70         4.20       3.20       25           Chen2020
LiFePO4              3.20         3.65       2.50       40           Marquis2019
Lithium NMC          3.70         4.20       3.00       25           Chen2020


In [7]:
# Chemistry thermal limits
print("\nCHEMISTRY THERMAL LIMITS")
print("=" * 70)
print(f"{'Chemistry':<20} {'Max Discharge Temp':<22} {'Min Temp'}")
print("-" * 70)

for name, chem in chemistries:
    max_t = chem.max_discharge_temperature.in_units_of('degC') if chem.max_discharge_temperature else 'N/A'
    min_t = chem.min_temperature.in_units_of('degC') if chem.min_temperature else 'N/A'
    print(f"{name:<20} {max_t} degC{'':<14} {min_t} degC")


CHEMISTRY THERMAL LIMITS
Chemistry            Max Discharge Temp     Min Temp
----------------------------------------------------------------------
Lithium-Ion          60 degC               -20 degC
Lithium Polymer      60 degC               -20 degC
LiFePO4              60 degC               -20 degC
Lithium NMC          60 degC               -20 degC


---
## Part 2: C-rate Conversions

C-rate represents current as a multiple of capacity. 1C on a 20Ah battery means 20A.

In [8]:
# Use our 14S4P battery from earlier
print(f"Battery: {battery.cells_series}S{battery.cells_parallel}P")
print(f"Total Capacity: {battery.total_capacity}")
print()

# C-rate to current
print("C-RATE TO CURRENT CONVERSION")
print("=" * 40)
for c_rate in [0.5, 1.0, 2.0, 3.0, 5.0]:
    current = battery.current_from_c_rate(c_rate)
    print(f"  {c_rate:.1f}C  =>  {current.in_units_of('A'):.1f} A")

Battery: 14S4P
Total Capacity: 20 ampere_hour

C-RATE TO CURRENT CONVERSION
  0.5C  =>  10.0 A
  1.0C  =>  20.0 A
  2.0C  =>  40.0 A
  3.0C  =>  60.0 A
  5.0C  =>  100.0 A


In [9]:
# Current to C-rate
print("CURRENT TO C-RATE CONVERSION")
print("=" * 40)
for current_a in [10, 20, 40, 60, 100]:
    c_rate = battery.c_rate_from_current(Current(current_a, 'A'))
    print(f"  {current_a:>3} A  =>  {c_rate:.2f}C")

CURRENT TO C-RATE CONVERSION
   10 A  =>  0.50C
   20 A  =>  1.00C
   40 A  =>  2.00C
   60 A  =>  3.00C
  100 A  =>  5.00C


In [10]:
# Current limits from C-ratings
print("\nCURRENT LIMITS")
print("=" * 40)
print(f"Max Charge Rate:     {battery.c_rating_charge}C")
print(f"Max Charge Current:  {battery.max_charge_current}")
print(f"Max Discharge Rate:  {battery.c_rating_discharge}C")
print(f"Max Discharge Current: {battery.max_discharge_current}")
print()
print(f"Max Charge Power:    {battery.max_charge_power.in_units_of('kW'):.2f} kW")
print(f"Max Discharge Power: {battery.max_discharge_power.in_units_of('kW'):.2f} kW")


CURRENT LIMITS
Max Charge Rate:     1.0C
Max Charge Current:  20.0 ampere
Max Discharge Rate:  2.0C
Max Discharge Current: 40.0 ampere

Max Charge Power:    1.04 kW
Max Discharge Power: 2.07 kW


---
## Part 3: Discharge Voltage Modeling

Battery voltage decreases with:
- Lower State of Charge (SoC)
- Higher discharge current (C-rate) due to internal resistance

### 3.1 Voltage vs SoC at Different C-rates

In [11]:
# Cell voltage at different SoC and C-rates
print("CELL VOLTAGE vs SoC and C-RATE")
print("=" * 70)
print(f"{'SoC':<8}", end="")
c_rates = [0.1, 0.5, 1.0, 2.0, 3.0]
for c_rate in c_rates:
    print(f"{c_rate:.1f}C{'':<8}", end="")
print()
print("-" * 70)

for soc in [1.0, 0.8, 0.6, 0.4, 0.2, 0.1]:
    print(f"{soc:<8.1f}", end="")
    for c_rate in c_rates:
        v = battery.get_cell_voltage(soc=soc, c_rate=c_rate)
        print(f"{v.in_units_of('V'):<11.3f}", end="")
    print()

CELL VOLTAGE vs SoC and C-RATE
SoC     0.1C        0.5C        1.0C        2.0C        3.0C        
----------------------------------------------------------------------
1.0     4.185      4.125      4.050      3.900      3.750      
0.8     3.905      3.845      3.770      3.620      3.470      
0.6     3.625      3.565      3.490      3.340      3.190      
0.4     3.345      3.285      3.210      3.060      2.910      
0.2     3.065      3.005      2.930      2.800      2.800      
0.1     2.925      2.865      2.800      2.800      2.800      


In [12]:
# Pack voltage at different SoC and C-rates
print("\nPACK VOLTAGE (14S) vs SoC and C-RATE")
print("=" * 70)
print(f"{'SoC':<8}", end="")
for c_rate in c_rates:
    print(f"{c_rate:.1f}C{'':<8}", end="")
print()
print("-" * 70)

for soc in [1.0, 0.8, 0.6, 0.4, 0.2, 0.1]:
    print(f"{soc:<8.1f}", end="")
    for c_rate in c_rates:
        v = battery.get_voltage(soc=soc, c_rate=c_rate)
        print(f"{v.in_units_of('V'):<11.2f}", end="")
    print()


PACK VOLTAGE (14S) vs SoC and C-RATE
SoC     0.1C        0.5C        1.0C        2.0C        3.0C        
----------------------------------------------------------------------
1.0     58.59      57.75      56.70      54.60      52.50      
0.8     54.67      53.83      52.78      50.68      48.58      
0.6     50.75      49.91      48.86      46.76      44.66      
0.4     46.83      45.99      44.94      42.84      40.74      
0.2     42.91      42.07      41.02      39.20      39.20      
0.1     40.95      40.11      39.20      39.20      39.20      


### 3.2 Using Current Instead of C-rate

In [13]:
# get_voltage() accepts either c_rate or current
soc = 0.5

# Method 1: Using C-rate
v_by_c_rate = battery.get_voltage(soc=soc, c_rate=1.0)

# Method 2: Using current (20A = 1C on 20Ah battery)
v_by_current = battery.get_voltage(soc=soc, current=Current(20, 'A'))

print(f"At SoC = {soc}:")
print(f"  Using c_rate=1.0:           {v_by_c_rate}")
print(f"  Using current=20A:          {v_by_current}")
print(f"  Same result: {abs(v_by_c_rate.in_units_of('V') - v_by_current.in_units_of('V')) < 0.001}")

At SoC = 0.5:
  Using c_rate=1.0:           46.9 volt
  Using current=20A:          46.9 volt
  Same result: True


### 3.3 Discharge Model Types

In [14]:
# The default discharge model is AnalyticalDischargeModel
default_model = battery.discharge_model
print(f"Default model type: {type(default_model).__name__}")
print()

# The analytical model uses: V = V_oc(SoC) - I*R
print("Analytical Model Parameters:")
print(f"  V_max (full charge OCV): {default_model.v_max}")
print(f"  V_min (empty OCV):       {default_model.v_min}")
print(f"  V_nominal:               {default_model.v_nominal}")
print(f"  Internal Resistance:     {default_model.internal_resistance}")
print(f"  Cell Capacity:           {default_model.capacity_ah} Ah")

Default model type: AnalyticalDischargeModel

Analytical Model Parameters:
  V_max (full charge OCV): 4.2 volt
  V_min (empty OCV):       2.8 volt
  V_nominal:               3.7 volt
  Internal Resistance:     30 milliohm
  Cell Capacity:           5 Ah


In [15]:
# Create a lookup table model (faster for repeated lookups)
lookup_model = create_default_lookup_table('lithium_ion')

print(f"Lookup Table Model:")
print(f"  SoC points: {len(lookup_model.soc_points)} ({lookup_model.soc_points[0]:.2f} to {lookup_model.soc_points[-1]:.2f})")
print(f"  C-rate points: {lookup_model.c_rate_points}")
print(f"  Table shape: {lookup_model.voltage_data.shape}")

Lookup Table Model:
  SoC points: 21 (0.00 to 1.00)
  C-rate points: [0.1 0.2 0.5 1.  2.  3.  5. ]
  Table shape: (21, 7)


In [16]:
# You can set a custom discharge model on the battery
battery.set_discharge_model(lookup_model)
print(f"Battery now using: {type(battery.discharge_model).__name__}")

# Compare voltages
v_lookup = battery.get_cell_voltage(soc=0.5, c_rate=1.0)
print(f"Cell voltage at SoC=0.5, 1C: {v_lookup}")

# Reset to default (analytical)
battery.set_discharge_model(None)  # None resets to default
# Actually need to explicitly set back since _discharge_model is now the lookup
battery._discharge_model = None

Battery now using: LookupTableDischargeModel
Cell voltage at SoC=0.5, 1C: 3.35 volt


---
## Part 4: Internal Resistance and Heat Generation

Pack resistance is calculated from cell resistance and configuration:
- Series cells add resistance
- Parallel cells reduce resistance

### 4.1 Pack Resistance Calculation

In [17]:
# Cell resistance from chemistry
cell_r = battery.chemistry.internal_resistance
print(f"Cell Internal Resistance: {cell_r}")
print()

# Pack resistance: R_pack = (R_cell * N_series) / N_parallel
print(f"Pack Configuration: {battery.cells_series}S{battery.cells_parallel}P")
print(f"Pack Resistance:    {battery.internal_resistance}")
print()

# Verify calculation
expected_r = cell_r.in_units_of('ohm') * battery.cells_series / battery.cells_parallel
print(f"Verification: ({cell_r.in_units_of('mohm'):.0f} mohm * {battery.cells_series}) / {battery.cells_parallel} = {expected_r*1000:.1f} mohm")

Cell Internal Resistance: 30 milliohm

Pack Configuration: 14S4P
Pack Resistance:    0.105 ohm

Verification: (30 mohm * 14) / 4 = 105.0 mohm


In [18]:
# How configuration affects pack resistance
print("PACK RESISTANCE vs CONFIGURATION")
print("=" * 60)
print(f"{'Config':<12} {'Series':<10} {'Parallel':<10} {'Pack R (mohm)'}")
print("-" * 60)

configs = [(1, 1), (10, 1), (1, 10), (14, 4), (100, 10)]
cell_r_ohm = 0.030  # 30 mohm

for s, p in configs:
    pack_r = (cell_r_ohm * s / p) * 1000  # mohm
    print(f"{s}S{p}P{'':<8} {s:<10} {p:<10} {pack_r:.1f}")

PACK RESISTANCE vs CONFIGURATION
Config       Series     Parallel   Pack R (mohm)
------------------------------------------------------------
1S1P         1          1          30.0
10S1P         10         1          300.0
1S10P         1          10         3.0
14S4P         14         4          105.0
100S10P         100        10         300.0


### 4.2 Heat Generation (I²R Losses)

In [19]:
# Heat generation = I² * R
print(f"Pack Resistance: {battery.internal_resistance}")
print()
print("HEAT GENERATION vs CURRENT")
print("=" * 50)
print(f"{'Current (A)':<15} {'C-rate':<10} {'Heat (W)'}")
print("-" * 50)

for current_a in [10, 20, 40, 60, 80, 100]:
    current = Current(current_a, 'A')
    c_rate = battery.c_rate_from_current(current)
    heat = battery.heat_generation_rate(current)
    print(f"{current_a:<15} {c_rate:<10.1f}C {heat.in_units_of('W'):.1f}")

Pack Resistance: 0.105 ohm

HEAT GENERATION vs CURRENT
Current (A)     C-rate     Heat (W)
--------------------------------------------------
10              0.5       C 10.5
20              1.0       C 42.0
40              2.0       C 168.0
60              3.0       C 378.0
80              4.0       C 672.0
100             5.0       C 1050.0


In [20]:
# Heat generation scales with I²
heat_20a = battery.heat_generation_rate(Current(20, 'A'))
heat_40a = battery.heat_generation_rate(Current(40, 'A'))
heat_80a = battery.heat_generation_rate(Current(80, 'A'))

print("HEAT SCALING (I² relationship)")
print("=" * 50)
print(f"Heat at 20A: {heat_20a.in_units_of('W'):.1f} W (baseline)")
print(f"Heat at 40A: {heat_40a.in_units_of('W'):.1f} W ({heat_40a.in_units_of('W')/heat_20a.in_units_of('W'):.0f}x = 2² times baseline)")
print(f"Heat at 80A: {heat_80a.in_units_of('W'):.1f} W ({heat_80a.in_units_of('W')/heat_20a.in_units_of('W'):.0f}x = 4² times baseline)")

HEAT SCALING (I² relationship)
Heat at 20A: 42.0 W (baseline)
Heat at 40A: 168.0 W (4x = 2² times baseline)
Heat at 80A: 672.0 W (16x = 4² times baseline)


---
## Part 5: Charge Modeling

The charge model implements CC-CV (Constant Current - Constant Voltage) charging:
- **CC Phase**: Full current until transition SoC (typically 80%)
- **CV Phase**: Current tapers to maintain constant voltage

In [21]:
# Default charge model
charge_model = battery.charge_model
print(f"Charge Model: {type(charge_model).__name__}")
print(f"CC-CV Transition SoC: {charge_model.cc_cv_transition_soc}")
print(f"CV Taper Factor: {charge_model.cv_taper_factor}")
print(f"Base Efficiency: {charge_model.base_efficiency}")

Charge Model: SimpleChargeModel
CC-CV Transition SoC: 0.8
CV Taper Factor: 3.0
Base Efficiency: 0.95


### 5.1 CC-CV Charge Profile

In [22]:
# Charge current at different SoC levels
print("CC-CV CHARGE PROFILE (Max 1C)")
print("=" * 60)
print(f"{'SoC':<10} {'Phase':<12} {'C-rate':<12} {'Current (A)'}")
print("-" * 60)

for soc in [0.1, 0.3, 0.5, 0.7, 0.8, 0.85, 0.9, 0.95]:
    current = battery.get_charge_current(soc=soc)
    c_rate = battery.c_rate_from_current(current)
    phase = "CC" if soc < charge_model.cc_cv_transition_soc else "CV"
    print(f"{soc:<10.2f} {phase:<12} {c_rate:<12.3f} {current.in_units_of('A'):.2f}")

CC-CV CHARGE PROFILE (Max 1C)
SoC        Phase        C-rate       Current (A)
------------------------------------------------------------
0.10       CC           1.000        20.00
0.30       CC           1.000        20.00
0.50       CC           1.000        20.00
0.70       CC           1.000        20.00
0.80       CV           1.000        20.00
0.85       CV           0.472        9.45
0.90       CV           0.223        4.46
0.95       CV           0.105        2.11


### 5.2 Charging Efficiency

In [23]:
# Efficiency varies with SoC and C-rate
print("CHARGING EFFICIENCY vs SoC and C-RATE")
print("=" * 55)
print(f"{'SoC':<10}", end="")
c_rates = [0.5, 1.0, 2.0]
for c_rate in c_rates:
    print(f"{c_rate:.1f}C{'':<10}", end="")
print()
print("-" * 55)

for soc in [0.2, 0.4, 0.6, 0.8, 0.9]:
    print(f"{soc:<10.1f}", end="")
    for c_rate in c_rates:
        eff = battery.get_charge_efficiency(soc=soc, c_rate=c_rate)
        print(f"{eff*100:<13.1f}%", end="")
    print()

CHARGING EFFICIENCY vs SoC and C-RATE
SoC       0.5C          1.0C          2.0C          
-------------------------------------------------------
0.2       93.4         %92.4         %90.4         %
0.4       92.8         %91.8         %89.8         %
0.6       92.2         %91.2         %89.2         %
0.8       91.6         %90.6         %88.6         %
0.9       91.3         %90.3         %88.3         %


### 5.3 Charge Time Estimation

In [24]:
# Time to charge between different SoC levels
print("CHARGE TIME ESTIMATION (1C max)")
print("=" * 50)
print(f"{'From':<10} {'To':<10} {'Time (hours)':<15} {'Time (min)'}")
print("-" * 50)

charge_ranges = [
    (0.2, 0.8),   # Typical fast charge range
    (0.2, 0.9),   # Extended charge
    (0.1, 0.95),  # Near-full charge
    (0.5, 0.8),   # Quick top-up
]

for start, end in charge_ranges:
    time_hr = battery.time_to_charge(start_soc=start, end_soc=end)
    time_min = time_hr * 60
    print(f"{start*100:.0f}%{'':<7} {end*100:.0f}%{'':<7} {time_hr:<15.3f} {time_min:.1f}")

CHARGE TIME ESTIMATION (1C max)
From       To         Time (hours)    Time (min)
--------------------------------------------------
20%        80%        0.656           39.3
20%        90%        0.896           53.8
10%        95%        1.342           80.5
50%        80%        0.329           19.8


---
## Part 6: Thermal Framework

The thermal framework provides:
- Temperature limits for safe operation
- Power derating at extreme temperatures
- Simple thermal dynamics model

### 6.1 Thermal Limits

In [25]:
# Default thermal limits
limits = ThermalLimits()

print("DEFAULT THERMAL LIMITS")
print("=" * 50)
print(f"Max Charge Temperature:    {limits.max_charge_temp}")
print(f"Max Discharge Temperature: {limits.max_discharge_temp}")
print(f"Min Temperature:           {limits.min_temp}")
print(f"Derate Temperature:        {limits.derate_temp}")

DEFAULT THERMAL LIMITS
Max Charge Temperature:    45 degree_Celsius
Max Discharge Temperature: 60 degree_Celsius
Min Temperature:           -20 degree_Celsius
Derate Temperature:        40 degree_Celsius


In [26]:
# Check if temperatures are within limits
print("TEMPERATURE LIMIT CHECKS")
print("=" * 60)
print(f"{'Temperature':<18} {'Discharge OK':<15} {'Charge OK'}")
print("-" * 60)

test_temps = [-25, -20, 0, 25, 40, 45, 50, 60, 65]
for temp_c in test_temps:
    temp = Temperature(temp_c, 'degC')
    discharge_ok = limits.is_within_limits(temp, is_charging=False)
    charge_ok = limits.is_within_limits(temp, is_charging=True)
    print(f"{temp_c:>4} degC{'':<10} {str(discharge_ok):<15} {charge_ok}")

TEMPERATURE LIMIT CHECKS
Temperature        Discharge OK    Charge OK
------------------------------------------------------------
 -25 degC           False           False
 -20 degC           True            True
   0 degC           True            True
  25 degC           True            True
  40 degC           True            True
  45 degC           True            True
  50 degC           True            False
  60 degC           True            False
  65 degC           False           False


### 6.2 Power Derating

In [27]:
# Power derating at high temperatures
print("HIGH TEMPERATURE DERATING")
print("=" * 50)
print(f"Derate starts at: {limits.derate_temp}")
print(f"Full cutoff at:   {limits.max_discharge_temp}")
print()
print(f"{'Temperature':<15} {'Derate Factor':<18} {'Available Power'}")
print("-" * 50)

for temp_c in [25, 35, 40, 45, 50, 55, 60, 65]:
    temp = Temperature(temp_c, 'degC')
    factor = limits.get_derate_factor(temp)
    print(f"{temp_c:>4} degC{'':<7} {factor*100:>5.1f}%{'':<11} {factor*100:.0f}%")

HIGH TEMPERATURE DERATING
Derate starts at: 40 degree_Celsius
Full cutoff at:   60 degree_Celsius

Temperature     Derate Factor      Available Power
--------------------------------------------------
  25 degC        100.0%            100%
  35 degC        100.0%            100%
  40 degC        100.0%            100%
  45 degC         75.0%            75%
  50 degC         50.0%            50%
  55 degC         25.0%            25%
  60 degC          0.0%            0%
  65 degC          0.0%            0%


In [28]:
# Cold temperature derating
print("COLD TEMPERATURE DERATING")
print("=" * 50)
print(f"Full power above: 0 degC")
print(f"Full cutoff at:   {limits.min_temp}")
print()
print(f"{'Temperature':<15} {'Cold Derate':<18} {'Available Power'}")
print("-" * 50)

for temp_c in [10, 5, 0, -5, -10, -15, -20, -25]:
    temp = Temperature(temp_c, 'degC')
    factor = limits.get_cold_derate_factor(temp)
    print(f"{temp_c:>4} degC{'':<7} {factor*100:>5.1f}%{'':<11} {factor*100:.0f}%")

COLD TEMPERATURE DERATING
Full power above: 0 degC
Full cutoff at:   -20 degree_Celsius

Temperature     Cold Derate        Available Power
--------------------------------------------------
  10 degC        100.0%            100%
   5 degC        100.0%            100%
   0 degC        100.0%            100%
  -5 degC         75.0%            75%
 -10 degC         50.0%            50%
 -15 degC         25.0%            25%
 -20 degC          0.0%            0%
 -25 degC          0.0%            0%


### 6.3 Simple Thermal Model

In [29]:
# Create a thermal model for our battery
thermal_model = SimpleThermalModel(
    mass=battery.mass,
    specific_heat=1000,     # J/kg/K typical for Li-ion
    cooling_coefficient=10.0  # W/K, depends on cooling system
)

print("SIMPLE THERMAL MODEL")
print("=" * 50)
print(f"Battery Mass:        {thermal_model.mass}")
print(f"Specific Heat:       {thermal_model.specific_heat} J/kg/K")
print(f"Cooling Coefficient: {thermal_model.cooling_coefficient} W/K")
print()

# Thermal time constant
tau = thermal_model.mass.in_units_of('kg') * thermal_model.specific_heat / thermal_model.cooling_coefficient
print(f"Thermal Time Constant: {tau:.0f} seconds ({tau/60:.1f} minutes)")

SIMPLE THERMAL MODEL
Battery Mass:        3.22 kilogram
Specific Heat:       1000 J/kg/K
Cooling Coefficient: 10.0 W/K

Thermal Time Constant: 322 seconds (5.4 minutes)


In [30]:
# Temperature rise during discharge
initial_temp = Temperature(25, 'degC')
ambient_temp = Temperature(25, 'degC')
discharge_current = Current(40, 'A')  # 2C
heat_rate = battery.heat_generation_rate(discharge_current)

print(f"TEMPERATURE RISE DURING DISCHARGE")
print(f"=" * 50)
print(f"Discharge current: {discharge_current} (2C)")
print(f"Heat generation:   {heat_rate}")
print(f"Initial temp:      {initial_temp}")
print(f"Ambient temp:      {ambient_temp}")
print()
print(f"{'Time':<12} {'Temperature (degC)'}")
print("-" * 30)

for t_min in [0, 5, 10, 15, 20, 30, 45, 60]:
    if t_min == 0:
        temp = initial_temp
    else:
        temp = thermal_model.temperature_after(
            initial_temp=initial_temp,
            heat_rate=heat_rate,
            duration=Time(t_min, 'min'),
            ambient_temp=ambient_temp
        )
    print(f"{t_min:>3} min{'':<5} {temp.in_units_of('degC'):.1f}")

TEMPERATURE RISE DURING DISCHARGE
Discharge current: 40 ampere (2C)
Heat generation:   168.0 watt
Initial temp:      25 degree_Celsius
Ambient temp:      25 degree_Celsius

Time         Temperature (degC)
------------------------------
  0 min      25.0
  5 min      35.2
 10 min      39.2
 15 min      40.8
 20 min      41.4
 30 min      41.7
 45 min      41.8
 60 min      41.8


In [31]:
# Steady-state temperature
steady_temp = thermal_model.steady_state_temperature(heat_rate, ambient_temp)
print(f"\nSteady-state temperature: {steady_temp.in_units_of('degC'):.1f} degC")
print(f"Temperature rise:         {steady_temp.in_units_of('degC') - ambient_temp.in_units_of('degC'):.1f} degC")


Steady-state temperature: 41.8 degC
Temperature rise:         16.8 degC


---
## Part 7: Mission Simulation Example

Combining discharge curves, heat generation, and thermal modeling for a complete mission simulation.

In [32]:
# Simple mission: constant power hover
hover_power = Power(2, 'kW')  # 2 kW hover power
initial_soc = 1.0
initial_temp = Temperature(25, 'degC')
ambient_temp = Temperature(30, 'degC')  # Hot day
time_step_min = 5  # 5-minute time steps

print("MISSION SIMULATION: CONSTANT POWER HOVER")
print("=" * 80)
print(f"Hover Power: {hover_power}")
print(f"Initial SoC: {initial_soc*100:.0f}%")
print(f"Initial Temp: {initial_temp}")
print(f"Ambient Temp: {ambient_temp}")
print()
print(f"{'Time (min)':<12} {'SoC (%)':<10} {'Voltage (V)':<14} {'Current (A)':<14} {'Temp (degC)':<12} {'Status'}")
print("-" * 80)

soc = initial_soc
temp = initial_temp
t = 0

while soc > 0.2 and t <= 60:  # Stop at 20% SoC or 60 minutes
    # Get voltage at current SoC (estimate current from power)
    # Initial estimate: I = P / V_nominal
    v_estimate = battery.get_voltage(soc=soc, c_rate=0.5)  # Start with low C-rate estimate
    current_estimate = hover_power.in_units_of('W') / v_estimate.in_units_of('V')
    c_rate = current_estimate / battery.total_capacity.in_units_of('Ah')
    
    # Refine with actual voltage
    voltage = battery.get_voltage(soc=soc, c_rate=c_rate)
    current = hover_power.in_units_of('W') / voltage.in_units_of('V')
    
    # Check temperature limits
    derate = limits.get_derate_factor(temp)
    status = "OK" if derate > 0.95 else f"DERATE {derate*100:.0f}%"
    
    print(f"{t:<12} {soc*100:<10.1f} {voltage.in_units_of('V'):<14.1f} "
          f"{current:<14.1f} {temp.in_units_of('degC'):<12.1f} {status}")
    
    # Update for next time step
    if t > 0:
        # Energy consumed in time step
        energy_wh = hover_power.in_units_of('W') * time_step_min / 60
        soc_consumed = energy_wh / battery.energy_capacity.in_units_of('Wh')
        soc -= soc_consumed
        
        # Temperature update
        heat = battery.heat_generation_rate(Current(current, 'A'))
        temp = thermal_model.temperature_after(
            initial_temp=temp,
            heat_rate=heat,
            duration=Time(time_step_min, 'min'),
            ambient_temp=ambient_temp
        )
    
    t += time_step_min

print("-" * 80)
print(f"Mission ended at {t-time_step_min} minutes with {soc*100:.1f}% SoC remaining")

MISSION SIMULATION: CONSTANT POWER HOVER
Hover Power: 2 kilowatt
Initial SoC: 100%
Initial Temp: 25 degree_Celsius
Ambient Temp: 30 degree_Celsius

Time (min)   SoC (%)    Voltage (V)    Current (A)    Temp (degC)  Status
--------------------------------------------------------------------------------
0            100.0      55.2           36.3           25.0         OK
5            100.0      55.2           36.3           25.0         OK
10           83.9       51.8           38.6           36.4         OK
15           67.8       48.4           41.3           42.0         DERATE 90%
20           51.7       45.0           44.5           45.6         DERATE 72%
25           35.6       41.5           48.2           48.7         DERATE 56%
--------------------------------------------------------------------------------
Mission ended at 25 minutes with 19.6% SoC remaining


---
## Part 8: Advanced Topics

### 8.1 Array Operations

In [33]:
# Discharge models support array SoC inputs
soc_array = np.linspace(0.1, 1.0, 10)

# Get voltage curve at 1C
model = battery.discharge_model
voltages = model.get_voltage(soc=soc_array, c_rate=1.0)

print("VOLTAGE CURVE (1C)")
print("=" * 40)
for soc, v in zip(soc_array, voltages.magnitude):
    print(f"  SoC {soc*100:>5.1f}%: {v:.3f} V")

VOLTAGE CURVE (1C)
  SoC  10.0%: 2.800 V
  SoC  20.0%: 2.930 V
  SoC  30.0%: 3.070 V
  SoC  40.0%: 3.210 V
  SoC  50.0%: 3.350 V
  SoC  60.0%: 3.490 V
  SoC  70.0%: 3.630 V
  SoC  80.0%: 3.770 V
  SoC  90.0%: 3.910 V
  SoC 100.0%: 4.050 V


### 8.2 Loading Models by Chemistry

In [34]:
# Use load_discharge_model for quick model creation
chemistries = ['lithium_ion', 'lipo', 'lifepo4', 'nmc']

print("LOAD DISCHARGE MODEL BY CHEMISTRY")
print("=" * 60)

for chem in chemistries:
    model = load_discharge_model(chem)
    v = model.get_voltage(soc=0.5, c_rate=1.0)
    print(f"{chem:<15} => {type(model).__name__:<30} V@50%: {v.in_units_of('V'):.2f}V")

LOAD DISCHARGE MODEL BY CHEMISTRY
lithium_ion     => AnalyticalDischargeModel       V@50%: 3.35V
lipo            => AnalyticalDischargeModel       V@50%: 3.58V
lifepo4         => AnalyticalDischargeModel       V@50%: 2.88V
nmc             => AnalyticalDischargeModel       V@50%: 3.48V


### 8.3 Custom Charge Model

In [35]:
# Create custom charge model with different parameters
fast_charge_model = SimpleChargeModel(
    cc_cv_transition_soc=0.7,   # Earlier transition for battery longevity
    cv_taper_factor=4.0,        # Faster taper
    base_efficiency=0.92,       # Slightly lower efficiency at high rates
    efficiency_c_rate_factor=0.03,  # More efficiency loss per C-rate
)

# Set it on the battery
battery.set_charge_model(fast_charge_model)

print("FAST CHARGE PROFILE (Custom Model)")
print("=" * 50)
print(f"CC-CV transition: {fast_charge_model.cc_cv_transition_soc*100:.0f}% SoC")
print()
for soc in [0.2, 0.5, 0.7, 0.8, 0.9]:
    current = battery.get_charge_current(soc=soc)
    c_rate = battery.c_rate_from_current(current)
    eff = battery.get_charge_efficiency(soc=soc)
    print(f"  SoC {soc*100:>3.0f}%: {current.in_units_of('A'):.1f}A ({c_rate:.2f}C), Eff: {eff*100:.1f}%")

# Reset to default
battery._charge_model = None

FAST CHARGE PROFILE (Custom Model)
CC-CV transition: 70% SoC

  SoC  20%: 20.0A (1.00C), Eff: 88.4%
  SoC  50%: 20.0A (1.00C), Eff: 87.5%
  SoC  70%: 20.0A (1.00C), Eff: 86.9%
  SoC  80%: 5.3A (0.26C), Eff: 86.6%
  SoC  90%: 1.4A (0.07C), Eff: 86.3%


---
## Summary

This notebook demonstrated the complete battery modeling capabilities in evtol-tools:

### Battery Configuration
- `Battery(cells_series, cells_parallel, ...)` - Direct configuration
- `Battery.from_target_voltage(...)` - Size from voltage requirement
- `Battery.from_target_energy(...)` - Size from energy requirement

### Chemistry Support
- Pre-defined: `LITHIUM_ION`, `LITHIUM_POLYMER`, `LITHIUM_IRON_PHOSPHATE`, `LITHIUM_NMC`
- Each includes: voltage limits, internal resistance, thermal limits, PyBaMM parameter set

### Discharge Modeling
- `battery.get_voltage(soc, c_rate)` - Pack voltage at operating conditions
- `battery.get_cell_voltage(soc, c_rate)` - Single cell voltage
- `AnalyticalDischargeModel` - Simple V = V_oc - I*R model
- `LookupTableDischargeModel` - 2D interpolation tables

### C-rate Conversions
- `battery.current_from_c_rate(c_rate)` - C-rate to current
- `battery.c_rate_from_current(current)` - Current to C-rate

### Thermal Analysis
- `battery.internal_resistance` - Pack resistance
- `battery.heat_generation_rate(current)` - I²R losses
- `ThermalLimits` - Temperature limits and derating
- `SimpleThermalModel` - Lumped-capacitance thermal dynamics

### Charge Modeling
- `SimpleChargeModel` - CC-CV charging profile
- `battery.get_charge_current(soc)` - Charge current from profile
- `battery.get_charge_efficiency(soc, c_rate)` - Charging efficiency
- `battery.time_to_charge(start_soc, end_soc)` - Charge time estimate